# Week 3: Pandas Data Cleaning Demo

**Learning Goals:**
- Load and inspect messy historical data
- Identify common data quality issues
- Apply systematic cleaning techniques
- Make informed decisions about data transformation
- Document cleaning choices

## Setup

In [86]:
import pandas as pd
import re

# Display settings for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Step 1: Load and Inspect Data

First, let's load our historical dataset and see what we're working with.

In [87]:
# Load the data
df = pd.read_csv('./week03-census-sample.csv')

# Display all rows with all columns side by side
print("All rows:")
print(df.to_string())

All rows:
                    Name Birth Year               Location             Occupation                 Notes
0     Frederick Douglass       1818       Talbot County MD    abolitionist writer       escaped slavery
1         Harriet Tubman    c. 1822  Dorchester County  MD              conductor  Underground Railroad
2        Sojourner Truth      1797?               New York  Preacher and activist                   NaN
3              S. Truth        1797               new york               preacher            duplicate?
4     Frederick douglass       1818          Talbot County                 Writer    possible duplicate
5           Ida B. Wells       1862       Holly Springs MS             journalist                   NaN
6         W.E.B. Du Bois       1868    Great Barrington MA  sociologist historian                   NaN
7   Booker T. Washington      1856?         Hale's Ford VA               educator         born enslaved
8    Mary Church Terrell       1863             Memphi

Note that the dataset contains various inconsistencies and missing values that we will need to address. Note also that some cells have 'NaN' as a string, which is different from actual missing values represented by `NaN` in pandas.

In [88]:
# Check the shape (rows, columns)
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Dataset shape: (9, 5)
Rows: 9, Columns: 5


In [89]:
# Get column information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Name        9 non-null      str  
 1   Birth Year  9 non-null      str  
 2   Location    9 non-null      str  
 3   Occupation  9 non-null      str  
 4   Notes       5 non-null      str  
dtypes: str(5)
memory usage: 492.0 bytes


**Observations:**
- All columns are 'object' type (strings)
- Birth Year should probably be numeric
- Some columns might have missing values

## Step 2: Identify Data Quality Issues

Let's systematically check for problems in our data.

Let's check for whitespace issues in string columns.

In [90]:
# Check for whitespace issues in string columns using pandas
# make sure to use pandas string methods
for col in df.select_dtypes(include=['object']).columns:
    if df[col].str.contains(r'^\s+|\s+$', na=False).any():
        print(f"Column '{col}' has leading or trailing whitespace.")


Column 'Name' has leading or trailing whitespace.


/tmp/ipykernel_94879/2753268150.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [91]:
# remove whitespace from column Name; also count the number of affected rows
affected_rows = df['Name'].str.contains(r'^\s+|\s+$', na=False).sum()
df['Name'] = df['Name'].str.strip()
# Verify removal
if not df['Name'].str.contains(r'^\s+|\s+$', na=False).any():
    print(f"Whitespace successfully removed from 'Name' column. Affected rows: {affected_rows}")

Whitespace successfully removed from 'Name' column. Affected rows: 2


In [92]:
# Check for missing values
print("Missing values per column:")
df.isnull().sum()

Missing values per column:


Name          0
Birth Year    0
Location      0
Occupation    0
Notes         4
dtype: int64

In [93]:
# Examine Birth Year values
print("Unique Birth Year values:")
print(df['Birth Year'].unique())

Unique Birth Year values:
<StringArray>
['1818', 'c. 1822', '1797?', '1797', '1862', '1868', '1856?', '1863']
Length: 8, dtype: str


In [94]:
# Check Location formatting
print("Unique Location values:")
print(df['Location'].unique())

Unique Location values:
<StringArray>
[     'Talbot County MD', 'Dorchester County  MD',              'New York',
              'new york',         'Talbot County',      'Holly Springs MS',
   'Great Barrington MA',        'Hale's Ford VA',            'Memphis TN']
Length: 9, dtype: str


In [95]:
# Look for potential duplicates
print("All names:")
print(df['Name'].values)

All names:
<StringArray>
[  'Frederick Douglass',       'Harriet Tubman',      'Sojourner Truth',
             'S. Truth',   'Frederick douglass',         'Ida B. Wells',
       'W.E.B. Du Bois', 'Booker T. Washington',  'Mary Church Terrell']
Length: 9, dtype: str


**Issues Identified:**
1. Birth Year contains uncertainty markers: "c. " (circa) and "?"
2. Location has inconsistent capitalization and extra spaces
3. Potential duplicate entries (Frederick Douglass, Sojourner Truth)
4. Missing value in Notes column

## Step 3: Clean Column Names

Make column names easier to work with: lowercase, underscores instead of spaces.

In [96]:
# Clean column names programmatically
df.columns = df.columns.str.lower().str.replace(' ', '_')

print("New column names:")
print(df.columns.tolist())

New column names:
['name', 'birth_year', 'location', 'occupation', 'notes']


## Step 4: Clean Birth Year Column

Remove uncertainty markers and convert to numeric type.

**Historical Decision:** We're removing information about uncertainty. In a real project, consider creating a separate `birth_year_certain` boolean column!

In [97]:
# The pandas way - using built-in string methods
df['birth_year'] = (df['birth_year']
    .astype(str)                       # Ensure it's string type    
    .str.replace('c. ', '', regex=False)  # Remove 'c. '
    .str.replace('?', '', regex=False)     # Remove '?'
    .str.strip()                           # Remove whitespace
)

# Convert to numeric, coercing errors to NaN
df['birth_year'] = pd.to_numeric(df['birth_year'], errors='coerce')

# Convert to nullable integer
df['birth_year'] = df['birth_year'].astype('Int64')

# Check the cleaned Birth Year values
print("Cleaned Birth Year values:")
print(df['birth_year'].unique())

Cleaned Birth Year values:
<IntegerArray>
[1818, 1822, 1797, 1862, 1868, 1856, 1863]
Length: 7, dtype: Int64


In [98]:
# See the cleaned dataset
print("Cleaned dataset:")
print(df.to_string())

Cleaned dataset:
                   name  birth_year               location             occupation                 notes
0    Frederick Douglass        1818       Talbot County MD    abolitionist writer       escaped slavery
1        Harriet Tubman        1822  Dorchester County  MD              conductor  Underground Railroad
2       Sojourner Truth        1797               New York  Preacher and activist                   NaN
3              S. Truth        1797               new york               preacher            duplicate?
4    Frederick douglass        1818          Talbot County                 Writer    possible duplicate
5          Ida B. Wells        1862       Holly Springs MS             journalist                   NaN
6        W.E.B. Du Bois        1868    Great Barrington MA  sociologist historian                   NaN
7  Booker T. Washington        1856         Hale's Ford VA               educator         born enslaved
8   Mary Church Terrell        1863            

## Step 5: Standardize Location

Fix capitalization and remove extra whitespace.

In [99]:
# Standardize: title case, strip extra whitespace
# Note: .title() can incorrectly capitalize letters after apostrophes (e.g., "Hale's" -> "Hale'S")
# We'll fix this by converting to title case first, then lowercasing any letter after an apostrophe
df['location'] = (df['location']
    .str.strip()
    .str.title()
    .str.replace(r"'([A-Z])", lambda m: "'" + m.group(1).lower(), regex=True)
)

print("Standardized locations:")
print(df['location'].unique())

Standardized locations:
<StringArray>
[     'Talbot County Md', 'Dorchester County  Md',              'New York',
         'Talbot County',      'Holly Springs Ms',   'Great Barrington Ma',
        'Hale's Ford Va',            'Memphis Tn']
Length: 8, dtype: str


In [100]:
# Normalize two-letter state abbreviations and clean spacing in the Location column.
# - Convert any standalone two-letter token (case-insensitive) to uppercase (e.g., "md" -> "MD")
# - Collapse multiple internal spaces and strip leading/trailing whitespace
# This preserves other words (e.g., "Talbot County MD" -> "Talbot County MD").

# Snapshot before changes (for quick comparison)
_before = df['location'].unique()

# Uppercase standalone two-letter tokens (case-insensitive)
# Use a raw string (r'...') so backslashes (like \b for word-boundary) are passed literally to the regex engine.
# Uppercase standalone two-letter tokens (case-insensitive)
df['location'] = df['location'].str.replace(
    r'\b([a-z]{2})\b',
    lambda m: m.group(1).upper(),
    regex=True,
    flags=re.IGNORECASE  # re.IGNORECASE makes the pattern case-insensitive (matches 'md','Md','MD', etc.)
)

# Normalize whitespace
df['location'] = df['location'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Show result
print("Locations before:")
print(_before)
print("\nLocations after:")
print(df['location'].unique())

Locations before:
<StringArray>
[     'Talbot County Md', 'Dorchester County  Md',              'New York',
         'Talbot County',      'Holly Springs Ms',   'Great Barrington Ma',
        'Hale's Ford Va',            'Memphis Tn']
Length: 8, dtype: str

Locations after:
<StringArray>
[    'Talbot County MD', 'Dorchester County MD',             'New York',
        'Talbot County',     'Holly Springs MS',  'Great Barrington MA',
       'Hale's Ford VA',           'Memphis TN']
Length: 8, dtype: str


In [101]:
# View the cleaned location data
df[['name', 'location']].head(10)

,name,location
0,Frederick Douglass,Talbot County MD
1,Harriet Tubman,Dorchester County MD
2,Sojourner Truth,New York
3,S. Truth,New York
4,Frederick douglass,Talbot County
5,Ida B. Wells,Holly Springs MS
6,W.E.B. Du Bois,Great Barrington MA
7,Booker T. Washington,Hale's Ford VA
8,Mary Church Terrell,Memphis TN


Let's put the state info in a new column for easier analysis later.

In [ ]:
# Put state info in a new column for easier analysis later
df['state'] = df['location'].str.extract(r'\b([A-Z]{2})\b', expand=False)

# print df as string
print(df.to_string())

                   name  birth_year              location             occupation                 notes state
0    Frederick Douglass        1818      Talbot County MD    abolitionist writer       escaped slavery    MD
1        Harriet Tubman        1822  Dorchester County MD              conductor  Underground Railroad    MD
2       Sojourner Truth        1797              New York  Preacher and activist                   NaN   NaN
3              S. Truth        1797              New York               preacher            duplicate?   NaN
4    Frederick douglass        1818         Talbot County                 Writer    possible duplicate   NaN
5          Ida B. Wells        1862      Holly Springs MS             journalist                   NaN    MS
6        W.E.B. Du Bois        1868   Great Barrington MA  sociologist historian                   NaN    MA
7  Booker T. Washington        1856        Hale's Ford VA               educator         born enslaved    VA
8   Mary Church Ter

## Step 6: Handle Duplicates

This requires historical judgment! Let's investigate potential duplicates.

In [117]:
# split name into first and last name columns, handling middle names/initials
# all values in the name columns will be lower case
name_split = df['name'].str.strip().str.lower().str.split(' ', n=2, expand=True)
df['first_name'] = name_split[0]
df['last_name'] = name_split.iloc[:, -1].combine_first(name_split.iloc[:, 1])
# split name into first and last name columns, handling middle names/initials
name_split = df['name'].str.strip().str.lower().str.split(' ', n=2, expand=True)
df['first_name'] = name_split[0]
df['last_name'] = name_split[2].combine_first(name_split[1])
df['middle_name'] = name_split[1].where(name_split[2].notna())
# print df as string
print(df.to_string())
print(df.to_string())

                   name  birth_year              location             occupation                 notes state            name_lower              name_tokens   last_name  potential_name_word_duplicates first_name middle_name
7  Booker T. Washington        1856        Hale's Ford VA               educator         born enslaved    VA  booker t. washington     {washington, booker}  washington                           False     booker          t.
0    Frederick Douglass        1818      Talbot County MD    abolitionist writer       escaped slavery    MD    frederick douglass    {douglass, frederick}    douglass                            True  frederick         NaN
4    Frederick douglass        1818         Talbot County                 Writer    possible duplicate   NaN    frederick douglass    {douglass, frederick}    douglass                            True  frederick         NaN
1        Harriet Tubman        1822  Dorchester County MD              conductor  Underground Railroad    MD

In [118]:
# Find rows that share any first_name, middle_name, or last_name value
# (excluding NaN/null/empty values)
from collections import defaultdict

name_columns = ['first_name', 'middle_name', 'last_name']
value_to_indices = defaultdict(set)

# Build a mapping of each name value to the row indices that have it
for col in name_columns:
    s = df[col]
    non_blank = s.notna() & (s.astype(str).str.strip() != '')
    
    for idx in df[non_blank].index:
        val = s[idx]
        if pd.notna(val) and str(val).strip():
            value_to_indices[val].add(idx)

# Find indices that share at least one name value
indices_with_shared_names = set()
for val, indices in value_to_indices.items():
    if len(indices) > 1:  # Value appears in multiple rows
        indices_with_shared_names.update(indices)

# Group rows by their shared connections
groups = defaultdict(set)
for val, indices in value_to_indices.items():
    if len(indices) > 1:
        for idx in indices:
            groups[idx].update(indices)

# Consolidate overlapping groups using union-find
def merge_groups(groups):
    """Merge groups that share any indices"""
    merged = []
    remaining = list(groups.values())
    
    while remaining:
        current = remaining.pop(0)
        changed = True
        while changed:
            changed = False
            new_remaining = []
            for other in remaining:
                if current & other:  # If groups overlap
                    current |= other
                    changed = True
                else:
                    new_remaining.append(other)
            remaining = new_remaining
        merged.append(current)
    
    return merged

consolidated = merge_groups(groups)

# Display results
print(f"Found {len(consolidated)} group(s) of rows sharing name values:\n")
for i, group in enumerate(consolidated, 1):
    sorted_indices = sorted(group)
    print(f"Group {i}: rows {sorted_indices}")
    display(df.loc[sorted_indices, ['name', 'first_name', 'middle_name', 'last_name', 'birth_year', 'location']].sort_values('name'))
    print()

Found 2 group(s) of rows sharing name values:

Group 1: rows [0, 4]


,name,first_name,middle_name,last_name,birth_year,location
0,Frederick Douglass,frederick,NaN,douglass,1818,Talbot County MD
4,Frederick douglass,frederick,NaN,douglass,1818,Talbot County



Group 2: rows [2, 3]


,name,first_name,middle_name,last_name,birth_year,location
3,S. Truth,s.,NaN,truth,1797,New York
2,Sojourner Truth,sojourner,NaN,truth,1797,New York


**Discussion Questions:**
- Are these the same person?
- Which record should we keep?
- What information would we lose by removing duplicates?

**Our Decision:** Remove entries that appear to be duplicates (rows 3 and 4)

In [120]:
# We need to assess the results and determine if we want to remove any of these rows as duplicates, or if they represent different individuals with shared name values.
# We can safely say that the rows with last_name "truth" and last_name "douglass" represent the same individuals. So we can remove one of the truth and one of the douglass.
# We might prefer to keep 0 and not 4; and to keep 2 and not 3.
# Let's remove rows 3 and 4, which are duplicates of rows 0 and 2, respectively.
df = df.drop(index=[3, 4]).reset_index(drop=True)
print(df.to_string())


                   name  birth_year              location             occupation                 notes state            name_lower            name_tokens   last_name  potential_name_word_duplicates first_name middle_name
0  Booker T. Washington        1856        Hale's Ford VA               educator         born enslaved    VA  booker t. washington   {washington, booker}  washington                           False     booker          t.
1    Frederick Douglass        1818      Talbot County MD    abolitionist writer       escaped slavery    MD    frederick douglass  {douglass, frederick}    douglass                            True  frederick         NaN
2        Harriet Tubman        1822  Dorchester County MD              conductor  Underground Railroad    MD        harriet tubman      {tubman, harriet}      tubman                           False    harriet         NaN
3       Sojourner Truth        1797              New York  Preacher and activist                   NaN   NaN       s

In [121]:
# View the final cleaned data
df[['name', 'birth_year', 'location', 'occupation']].head(10)

,name,birth_year,location,occupation
0,Booker T. Washington,1856,Hale's Ford VA,educator
1,Frederick Douglass,1818,Talbot County MD,abolitionist writer
2,Harriet Tubman,1822,Dorchester County MD,conductor
3,Sojourner Truth,1797,New York,Preacher and activist
4,W.E.B. Du Bois,1868,Great Barrington MA,sociologist historian


## Step 7: Export Cleaned Data

Save the cleaned dataset and document what we did.

In [123]:
# Drop the temporary helper columns; but let's keep the state column for analysis; safe: ignore if missing
df = df.drop(columns=['name_lower', 'first_name', 'last_name', 'middle_name', 'name_tokens', 'potential_name_word_duplicates'], errors='ignore')

# Export to CSV
df.to_csv('./week03-census-cleaned.csv', index=False)
print("Cleaned data saved to: ./week03-census-cleaned.csv")

Cleaned data saved to: ./week03-census-cleaned.csv


## Summary

### What We Did

1. ✅ Loaded and inspected the data
2. ✅ Identified data quality issues
3. ✅ Cleaned column names (lowercase, underscores)
4. ✅ Cleaned birth year values (removed uncertainty markers)
5. ✅ Standardized location formatting
6. ✅ Extracted state information into a new column
7. ✅ Identified and removed duplicate records
8. ✅ Exported cleaned data

### Important Takeaways

- **Cleaning is interpretation**: Every choice removes or transforms information
- **Document everything**: What changed? Why? What was lost?
- **Preserve original data**: Always keep raw data untouched
- **80% of data work is cleaning**: This is normal!
- **Use domain expertise**: Historical knowledge guides cleaning decisions

### Next Steps

1. Apply these techniques to another dataset
2. Create a `README.md` documenting your cleaning steps
3. Keep an `AI-use-log.md` if you use Copilot
4. Consider what information your cleaning removes
5. Apply new data cleaning steps to new datasets